In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [3]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str
    

In [4]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
from urllib3.util import response


def generate_explanation(state: JokeState):
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation' : response}

In [6]:
from langgraph.checkpoint.memory import InMemorySaver


graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)


check_pointer = InMemorySaver()

workflow = graph.compile(checkpointer=check_pointer)



In [7]:
config1 = {'configurable' : {'thread_id' : "1"}}
final_state = workflow.invoke({'topic':'pizza'}, config = config1)

print(final_state)



{'topic': 'pizza', 'joke': 'Why did the pizza maker go broke?\n\nBecause he just couldn’t make enough dough!', 'explanation': 'The joke "Why did the pizza maker go broke? Because he just couldn’t make enough dough!" plays on a pun involving the word "dough." \n\nIn one sense, "dough" refers to the mixture of flour, water, and other ingredients used to make pizza crust. This is a literal interpretation related to the pizza industry, where making plenty of dough (i.e., pizza) is crucial for success.\n\nHowever, the joke relies on a double meaning of "dough," which is also a slang term for money. When we say someone "couldn\'t make enough dough," it means they weren\'t earning enough money to stay financially afloat.\n\nSo, the humor arises from the clever wordplay. The pizza maker failed in business because he couldn\'t create enough pizza (dough) to generate revenue, thus leading to him going broke. The dual interpretation of "dough" is what makes the punchline amusing.'}


In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker go broke?\n\nBecause he just couldn’t make enough dough!', 'explanation': 'The joke "Why did the pizza maker go broke? Because he just couldn’t make enough dough!" plays on a pun involving the word "dough." \n\nIn one sense, "dough" refers to the mixture of flour, water, and other ingredients used to make pizza crust. This is a literal interpretation related to the pizza industry, where making plenty of dough (i.e., pizza) is crucial for success.\n\nHowever, the joke relies on a double meaning of "dough," which is also a slang term for money. When we say someone "couldn\'t make enough dough," it means they weren\'t earning enough money to stay financially afloat.\n\nSo, the humor arises from the clever wordplay. The pizza maker failed in business because he couldn\'t create enough pizza (dough) to generate revenue, thus leading to him going broke. The dual interpretation of "dough" is what makes the punchline amus

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker go broke?\n\nBecause he just couldn’t make enough dough!', 'explanation': 'The joke "Why did the pizza maker go broke? Because he just couldn’t make enough dough!" plays on a pun involving the word "dough." \n\nIn one sense, "dough" refers to the mixture of flour, water, and other ingredients used to make pizza crust. This is a literal interpretation related to the pizza industry, where making plenty of dough (i.e., pizza) is crucial for success.\n\nHowever, the joke relies on a double meaning of "dough," which is also a slang term for money. When we say someone "couldn\'t make enough dough," it means they weren\'t earning enough money to stay financially afloat.\n\nSo, the humor arises from the clever wordplay. The pizza maker failed in business because he couldn\'t create enough pizza (dough) to generate revenue, thus leading to him going broke. The dual interpretation of "dough" is what makes the punchline amu

In [12]:
#Time Travel

workflow.get_state({'configurable' : {"thread_id" : "1", "checkpoint_id": "1f1e0009-ca8d-6354-8000-965247e54344"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1e0009-ca8d-6354-8000-965247e54344'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-11-14T07:54:25.812734+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e0009-ca7f-6326-bfff-6cc6fe2e028f'}}, tasks=(PregelTask(id='6ec1119d-e7c1-bc27-7ce5-98f9ca8a8b93', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza maker go broke?\n\nBecause he just couldn’t make enough dough!'}),), interrupts=())

In [13]:
workflow.invoke(None, {'configurable' : {"thread_id" : "1", "checkpoint_id": "1f1e0009-ca8d-6354-8000-965247e54344"}})

{'topic': 'pizza',
 'joke': "Why did the pizza maker go broke? \n\nBecause he just couldn't make enough dough! 🍕",
 'explanation': 'The joke "Why did the pizza maker go broke? Because he just couldn\'t make enough dough!" is a play on words involving a double meaning of the term "dough." \n\n1. **Literal Meaning**: In the context of pizza making, "dough" refers to the mixture of flour, water, and other ingredients that is used to create the base of the pizza. A pizza maker needs to make a good amount of dough to produce pizzas.\n\n2. **Figurative Meaning**: "Dough" is also a slang term for money. When the joke says the pizza maker "couldn\'t make enough dough," it implies that he wasn\'t earning enough money to keep his business running. \n\nThe humor arises from the clever wordplay that connects these two meanings of "dough," creating a light-hearted and punny punchline.'}

Update State


In [15]:
# Time travel: fork the run from an earlier checkpoint and change the topic
new_config = workflow.update_state(
    {"configurable": {"thread_id": "1", "checkpoint_ns": "", "checkpoint_id": "1f1e0009-ca8d-6354-8000-965247e54344"}},
    {"topic": "samosa"},
)

print(new_config)


{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e012d-b914-608a-8001-876275d88d7c'}}


In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e012d-b914-608a-8001-876275d88d7c'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-11-14T10:05:02.295810+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e0009-ca8d-6354-8000-965247e54344'}}, tasks=(PregelTask(id='3aebb201-2ef9-d8da-5fe0-0dff7e43fc6a', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go broke? \n\nBecause he just couldn't make enough dough! 🍕", 'explanation': 'The joke "Why did the pizza maker go broke? Because he just couldn\'t make enough dough!" is a play on words involving a double meaning of the term "dough." \n\n1. **Literal Meaning**: In the context of pizza making, "dough" ref

In [17]:
workflow.invoke(None, {'configurable' : {"thread_id" : "1", "checkpoint_id": "1f1e012d-b914-608a-8001-876275d88d7c"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to school?\n\nBecause it wanted to be a little more filling!',
 'explanation': 'The joke "Why did the samosa go to school? Because it wanted to be a little more filling!" is a play on words that uses a pun to create humor. \n\n1. **Samosa Background**: A samosa is a popular Indian snack that is typically filled with a mixture of potatoes, peas, spices, and sometimes meat. The term "filling" in this context has a double meaning.\n\n2. **Dual Meaning of "Filling"**: The word "filling" can refer to both the contents inside a samosa (what makes it satisfying to eat) and to the idea of being more educated or knowledgeable (as in "filling one\'s mind" with information).\n\n3. **Setup and Punchline**: The setup of the joke leads the listener to think about a samosa in a literal sense—food going to school—which is an absurd image. The punchline reveals that the samosa\'s goal is to enhance its "filling" in a figurative sense, making it both a